In [3]:
# Libraries to import:
import pandas as pd
import datetime
import webbrowser
import math
import folium
from haversine import haversine

# Custom class which stores address points:
class MapPoints:

    def __init__ (self, zipcode, coordinates, maxdistance_km, region):
        self.zipcode = zipcode
        self.coordinates = coordinates
        self.maxdistance_km = maxdistance_km
        self.region = region

# This function converts the zip code from float to integer:
def ConvertZipCodeToInteger(row):
    val = 0
    if pd.notnull(row['Zipcode']):
        val = int(row['Zipcode'])
        
    return val

# This function merged all the address fields:
def CalcFullAddress(row):

    FullAddress = ""

    if pd.notnull(row['PrefixAddressNumber']):
        FullAddress = FullAddress + row['PrefixAddressNumber'] + " "

    if pd.notnull(row['AddressNumber']):
        FullAddress = FullAddress + str(row['AddressNumber']) + " "

    if pd.notnull(row['SuffixAddressNumber']):
        FullAddress = FullAddress + row['SuffixAddressNumber'] + " "
        
    if pd.notnull(row['CompleteStreetName']):
        FullAddress = FullAddress + row['CompleteStreetName'] + ", "

    if pd.notnull(row['CityTownName']):
        FullAddress = FullAddress + row['CityTownName'] + ", "

    if pd.notnull(row['State']):
        FullAddress = FullAddress + row['State'] + ", "

    if pd.notnull(row['ZipCodeInteger']):
        FullAddress = FullAddress + str(row['ZipCodeInteger']) + ", United States"
    
    return FullAddress

def CalculateCentralPoint():

    print (datetime.datetime.now())
    print ("Calculating distance between points...")
    
    # First calculate the distance between points:  
    for Point1 in AllMapPoints:

        MaxDistance = 0
        
        for Point2 in AllMapPoints:
        
            if Point1.coordinates != None and Point2.coordinates != None:
            
                distance = haversine (Point1.coordinates, Point2.coordinates)
    
                if distance > 0 and distance > MaxDistance:
                    MaxDistance = distance
    
        Point1.maxdistance_km = MaxDistance
    
    # Calculating central point:
    CentralPoint_Zipcode = ""
    CentralPoint_Coordinates = ()
    CentralPoint_Distance = 999999
    
    for point in AllMapPoints:
        
        if point.maxdistance_km > 0 and  point.maxdistance_km < CentralPoint_Distance:
            
            CentralPoint_Zipcode = point.zipcode
            CentralPoint_Coordinates = point.coordinates
            CentralPoint_Distance = point.maxdistance_km
    
    return (CentralPoint_Coordinates, CentralPoint_Distance, CentralPoint_Zipcode)

def CalculateCentralPoint4Region (myregion):

    print (datetime.datetime.now())
    print ("Calculating distance between points...")
    
    # First calculate the distance between points:  
    for Point1 in AllMapPoints:

        if Point1.region == myregion:
            
            MaxDistance = 0
            
            for Point2 in AllMapPoints:

                if Point2.region == myregion:
            
                    if Point1.coordinates != None and Point2.coordinates != None:
                    
                        distance = haversine (Point1.coordinates, Point2.coordinates)
            
                        if distance > 0 and distance > MaxDistance:
                            MaxDistance = distance
        
            Point1.maxdistance_km = MaxDistance
    
    # Calculating central point:
    CentralPoint4Region_Zipcode = ""
    CentralPoint4Region_Coordinates = ()
    CentralPoint4Region_Distance = 999999
    
    for point in AllMapPoints:

        if point.region == myregion:
        
            if point.maxdistance_km > 0 and  point.maxdistance_km < CentralPoint4Region_Distance:
                
                CentralPoint4Region_Zipcode = point.zipcode
                CentralPoint4Region_Coordinates = point.coordinates
                CentralPoint4Region_Distance = point.maxdistance_km
    
    return (CentralPoint4Region_Coordinates, CentralPoint4Region_Distance, CentralPoint4Region_Zipcode)

###########################################################################################
# Main processing:
###########################################################################################

# Import zipcode geocoding file into data frame:
print (datetime.datetime.now())
print ("Importing the zip code geocoding file...")
df1 = pd.read_csv("zip_lat_long.csv", delimiter=',')

# Convert zipcode to integer, so we can join on zipcodes:
df1['ZIP'] = df1['ZIP'].astype(int)

# Import Nassau County, Long Island zip codes.
dfNassauCountyZipcodes = pd.read_csv("Zipcodes-NassauCountyLongIsland.csv")

# Convert zipcodes to Integer, so we can join to another data frame:
print (datetime.datetime.now())
print ("Converting zipcodes to integers...")
dfNassauCountyZipcodes['ZipCodeInteger'] = dfNassauCountyZipcodes.apply(ConvertZipCodeToInteger, axis=1)

# Geocode NY zipcodes by merging both data frames:
print (datetime.datetime.now())
print ("Geocoding zipcodes...")
dfNassauCountyZipcodes = dfNassauCountyZipcodes.merge(df1,left_on='ZipCodeInteger', right_on='ZIP')

# Load data into a list of custom coordinate class objects:
AllMapPoints = []
for index, row in dfNassauCountyZipcodes.iterrows():
    
    if pd.notnull(row['ZipCodeInteger']) and pd.notnull(row['LAT']) and pd.notnull(row['LNG']):
        # Create one instance of custom class:
        OneMapPoint = MapPoints (row['ZipCodeInteger'], (row['LAT'], row['LNG']), None,  "All")

        # Append instance to list of all map points:
        AllMapPoints.append (OneMapPoint)

# Calculate central point for all of Nassau County:
(CentralPoint_Coordinates, CentralPoint_Distance, CentralPoint_Zipcode) = CalculateCentralPoint()

# Use Central Point to split coordinates into North, South, East & West:
for Point in AllMapPoints:
    
    if Point.coordinates != None:
        
        if Point.coordinates[0] > CentralPoint_Coordinates[0] and Point.coordinates[1] > CentralPoint_Coordinates[1]:
            Point.region = 'NorthEast'
            
        elif Point.coordinates[0] > CentralPoint_Coordinates[0] and Point.coordinates[1] < CentralPoint_Coordinates[1]:
            Point.region = 'NorthWest'

        elif Point.coordinates[0] < CentralPoint_Coordinates[0] and Point.coordinates[1] < CentralPoint_Coordinates[1]:
            Point.region = 'SouthWest'

        elif Point.coordinates[0] < CentralPoint_Coordinates[0] and Point.coordinates[1] > CentralPoint_Coordinates[1]:
            Point.region = 'SouthEast'

# Calculate central point for NorthEast Nassau County:
(CentralPointNorthEast_Coordinates, CentralPointNorthEast_Distance, CentralPointNorthEast_Zipcode) = CalculateCentralPoint4Region("NorthEast")

# Calculate central point for NorthWest Nassau County:
(CentralPointNorthWest_Coordinates, CentralPointNorthWest_Distance, CentralPointNorthWest_Zipcode) = CalculateCentralPoint4Region("NorthWest")

# Calculate central point for SouthEast Nassau County:
(CentralPointSouthEast_Coordinates, CentralPointSouthEast_Distance, CentralPointSouthEast_Zipcode) = CalculateCentralPoint4Region("SouthEast")

# Calculate central point for SouthWest Nassau County:
(CentralPointSouthWest_Coordinates, CentralPointSouthWest_Distance, CentralPointSouthWest_Zipcode) = CalculateCentralPoint4Region("SouthWest")

# Define map:
map = folium.Map(location = [40.719678, -73.58386], zoom_start = 11)

# Add all coordinates to map:
for OnePoint in AllMapPoints:

    if OnePoint.coordinates != None and OnePoint.region == "NorthEast":
        folium.Marker (location=[OnePoint.coordinates[0],OnePoint.coordinates[1]], popup=OnePoint.maxdistance_km, icon=folium.Icon(color="blue")).add_to(map)

    elif OnePoint.coordinates != None and OnePoint.region == "NorthWest":
        folium.Marker (location=[OnePoint.coordinates[0],OnePoint.coordinates[1]], popup=OnePoint.maxdistance_km, icon=folium.Icon(color="blue")).add_to(map)

    elif OnePoint.coordinates != None and OnePoint.region == "SouthWest":
        folium.Marker (location=[OnePoint.coordinates[0],OnePoint.coordinates[1]], popup=OnePoint.maxdistance_km, icon=folium.Icon(color="blue")).add_to(map)

    elif OnePoint.coordinates != None and OnePoint.region == "SouthEast":
        folium.Marker (location=[OnePoint.coordinates[0],OnePoint.coordinates[1]], popup=OnePoint.maxdistance_km, icon=folium.Icon(color="blue")).add_to(map)

# Add central point to map:
#folium.Marker (location=[CentralPoint_Coordinates[0],CentralPoint_Coordinates[1]], popup=CentralPoint_Distance, icon=folium.Icon(color="red")).add_to(map)

# Add central point for NorthEast Nassau County to map:
folium.Marker (location=[CentralPointNorthEast_Coordinates[0],CentralPointNorthEast_Coordinates[1]], popup=CentralPointNorthEast_Distance, icon=folium.Icon(color="red")).add_to(map)

# Add central point for NorthWest Nassau County to map:
folium.Marker (location=[CentralPointNorthWest_Coordinates[0],CentralPointNorthWest_Coordinates[1]], popup=CentralPointNorthWest_Distance, icon=folium.Icon(color="red")).add_to(map)

# Add central point for SouthEast Nassau County to map:
folium.Marker (location=[CentralPointSouthEast_Coordinates[0],CentralPointSouthEast_Coordinates[1]], popup=CentralPointSouthEast_Distance, icon=folium.Icon(color="red")).add_to(map)

# Add central point for SouthWest Nassau County to map:
folium.Marker (location=[CentralPointSouthWest_Coordinates[0],CentralPointSouthWest_Coordinates[1]], popup=CentralPointSouthWest_Distance, icon=folium.Icon(color="red")).add_to(map)

# Save the map:
map.save("PythonOptimization-NassauCounty-CentralPointAlgorithm.html")
webbrowser.open_new_tab("PythonOptimization-NassauCounty-CentralPointAlgorithm.html")

print ("Completed")
print (datetime.datetime.now())


2025-05-29 13:31:34.010243
Importing the zip code geocoding file...
2025-05-29 13:31:34.022345
Converting zipcodes to integers...
2025-05-29 13:31:34.023560
Geocoding zipcodes...
2025-05-29 13:31:34.026664
Calculating distance between points...
2025-05-29 13:31:34.031043
Calculating distance between points...
2025-05-29 13:31:34.031434
Calculating distance between points...
2025-05-29 13:31:34.031607
Calculating distance between points...
2025-05-29 13:31:34.032011
Calculating distance between points...
Completed
2025-05-29 13:31:34.126600
